# Dispatch of a backtest run

Reads a run saved by `uv run backtest perfect-foresight` from `runs/<name>/`:
`summary.json` (`RunSummary`: totals, monthly and daily stats) and `dispatch.parquet` (one row per interval).

In [ ]:
from datetime import timedelta

import plotly.graph_objects as go
import polars as pl
from plotly.subplots import make_subplots

from battery_optimisation.bidding.strategy import Horizon
from battery_optimisation.columns import Col
from battery_optimisation.sim import backtest
from battery_optimisation.sim.backtest import StrategyName

runs = {backtest.load_run(p)[0].name: p for p in backtest.list_runs()}
list(runs)

In [ ]:
summary, dispatch = next(
    backtest.load_run(p)
    for p in runs.values()
    if (s := backtest.load_run(p)[0]).strategy is StrategyName.PERFECT_FORESIGHT
    and s.horizon is Horizon.FULL
    and not s.ancillary
)
t = summary.total
print(f"{summary.name}: {summary.days} days")
print(
    f"profit {t.profit_sgd:,.0f} SGD ({summary.profit_per_day_sgd:,.0f}/day), "
    f"degradation {t.degradation_sgd:,.0f} SGD, "
    f"{t.cycles:,.0f} cycles ({summary.cycles_per_day:.2f}/day)"
)
dispatch.head()

## Profit by month

The first and last months can be partial (the window is 12 months back from the last day of history).

In [ ]:
monthly = pl.DataFrame([m.model_dump() for m in summary.monthly])
fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_bar(x=monthly[Col.PERIOD_START], y=monthly[Col.PROFIT_SGD], name="profit (SGD)")
fig.add_scatter(
    x=monthly[Col.PERIOD_START],
    y=monthly[Col.MEAN_USEP],
    name="mean USEP (SGD/MWh)",
    secondary_y=True,
)
fig.update_layout(title="Monthly profit and mean price", hovermode="x unified")

## Daily profit and cycles

With the full-horizon benchmark, energy bought on one day can be sold on a later day, so some days show a negative cash profit. The daily-horizon run (`--horizon day`) forces each day to end at the initial SOC.

In [ ]:
daily = pl.DataFrame([d.model_dump() for d in summary.daily])
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.05)
fig.add_bar(x=daily[Col.PERIOD_START], y=daily[Col.PROFIT_SGD], name="profit (SGD)", row=1, col=1)
fig.add_scatter(
    x=daily[Col.PERIOD_START], y=daily[Col.CYCLES], name="cycles", mode="lines", row=2, col=1
)
fig.update_layout(
    title="Daily profit and equivalent full cycles", height=550, hovermode="x unified"
)

## Dispatch profile

Net power (bars: discharge up, charge down) with USEP overlaid, and SOC below. Defaults to the 3 days around the most profitable day; set `start` / `days` to look elsewhere.

In [ ]:
best_day = daily.sort(Col.PROFIT_SGD)[-1, Col.PERIOD_START]
start = best_day - timedelta(days=1)  # or e.g. date(2026, 3, 2)
days = 3

window = dispatch.filter(
    pl.col(Col.INTERVAL_START).dt.date().is_between(start, start + timedelta(days=days - 1))
)
x = window[Col.INTERVAL_START]
net = window[Col.NET_MW]

fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.06,
    row_heights=[0.65, 0.35],
    specs=[[{"secondary_y": True}], [{}]],
)
fig.add_bar(
    x=x,
    y=net,
    name="net MW (+ discharge)",
    row=1,
    col=1,
    marker_color=["#2a9d8f" if v >= 0 else "#e76f51" for v in net],
)
fig.add_scatter(
    x=x,
    y=window[Col.USEP],
    name="USEP (SGD/MWh)",
    mode="lines",
    line_color="#264653",
    row=1,
    col=1,
    secondary_y=True,
)
fig.add_scatter(
    x=x,
    y=window[Col.SOC_MWH],
    name="SOC (MWh)",
    mode="lines",
    fill="tozeroy",
    line_color="#8d99ae",
    row=2,
    col=1,
)
fig.update_yaxes(title_text="MW", row=1, col=1)
fig.update_yaxes(title_text="SGD/MWh", row=1, col=1, secondary_y=True)
fig.update_yaxes(title_text="MWh", range=[0, summary.battery.e_max_mwh], row=2, col=1)
fig.update_layout(
    title=f"Dispatch {start} (+{days} days)", height=600, hovermode="x unified", bargap=0
)

## Full vs daily horizon

Only if both runs exist (`uv run backtest perfect-foresight --horizon day`).

In [ ]:
by_horizon = {
    s.horizon: s
    for s, _ in (backtest.load_run(p) for p in runs.values())
    if s.strategy is StrategyName.PERFECT_FORESIGHT
    and not s.ancillary
    and (s.start, s.end) == (summary.start, summary.end)
}
if Horizon.DAY in by_horizon:
    full, day = by_horizon[Horizon.FULL], by_horizon[Horizon.DAY]
    share = day.total.profit_sgd / full.total.profit_sgd
    print(f"daily horizon captures {share:.1%} of the full-horizon optimum")
    cmp = pl.DataFrame(
        {
            Col.PERIOD_START: [m.period_start for m in full.monthly],
            Horizon.FULL: [m.profit_sgd for m in full.monthly],
            Horizon.DAY: [m.profit_sgd for m in day.monthly],
        }
    )
    display(
        go.Figure(
            [go.Bar(x=cmp[Col.PERIOD_START], y=cmp[h], name=h) for h in (Horizon.FULL, Horizon.DAY)]
        ).update_layout(title="Monthly profit by horizon (SGD)", barmode="group")
    )